#### Topic 4: Aggregations
  - Aggregate functions: COUNT, SUM, AVG, MIN, MAX, etc.
  - GROUP BY / HAVING
  - ROLLUP  (subtotals along hierarchy)
  - CUBE    (all combinations of subtotals)
  - GROUPING SETS (manual combinations)
  - GROUPING() helper function
  - PIVOT / UNPIVOT
  - COUNT DISTINCT vs APPROX_COUNT_DISTINCT

In [0]:
"""
QUICK REFERENCE:
─────────────────────────────────────────────────────────────────────
COUNT(*) counts ALL rows including NULLs
COUNT(col) counts non-NULL values only
COUNT(DISTINCT col) counts distinct non-NULL values

ROLLUP(a,b) → (a,b) + (a) + ()            [n+1 levels]
CUBE(a,b)   → (a,b) + (a) + (b) + ()      [2^n combinations]
GROUPING SETS → you control exactly which combos

GROUPING(col) = 1 → NULL is from ROLLUP/CUBE (subtotal)
GROUPING(col) = 0 → real data (might still be NULL value)

HAVING filters AFTER aggregation (post-GROUP BY)
WHERE  filters BEFORE aggregation (pre-GROUP BY)
You cannot use aggregate functions in WHERE clause.

COLLECT_LIST → preserves duplicates
COLLECT_SET  → removes duplicates
─────────────────────────────────────────────────────────────────────
"""


##### 1. CORE AGGREGATE FUNCTIONS

In [0]:
from pyspark.sql import SparkSession

# ---------------------------------------------------------------------------
# Sample data
# ---------------------------------------------------------------------------
spark.sql("""
    CREATE OR REPLACE TEMP VIEW sales AS
    SELECT * FROM VALUES
        ('Q1', 'Engineering', 'Alice',   95000),
        ('Q1', 'Engineering', 'Carol',  110000),
        ('Q1', 'Marketing',   'Bob',     72000),
        ('Q1', 'Marketing',   'Eve',     85000),
        ('Q1', 'HR',          'Dave',    60000),
        ('Q2', 'Engineering', 'Alice',   98000),
        ('Q2', 'Engineering', 'Frank',   98000),
        ('Q2', 'Marketing',   'Bob',     74000),
        ('Q2', 'HR',          'Dave',    62000),
        ('Q2', 'HR',          'Grace',   63000)
    AS t(quarter, department, employee, salary)
""")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW orders AS
    SELECT * FROM VALUES
        (1, 'Alice',  'Laptop',   2, 999.99,  '2024-01-10'),
        (2, 'Alice',  'Mouse',    3, 29.99,   '2024-01-15'),
        (3, 'Bob',    'Keyboard', 1, 79.99,   '2024-01-10'),
        (4, 'Bob',    'Monitor',  2, 399.99,  '2024-02-05'),
        (5, 'Carol',  'Laptop',   1, 999.99,  '2024-02-10'),
        (6, 'Carol',  'Headset',  4, 149.99,  '2024-02-20'),
        (7, 'Dave',   'Mouse',    5, 29.99,   '2024-03-01'),
        (8, NULL,     'Keyboard', 2, 79.99,   '2024-03-05')
    AS t(order_id, customer, product, qty, unit_price, order_date)
""")

# ---------------------------------------------------------------------------
# 1. CORE AGGREGATE FUNCTIONS
# ---------------------------------------------------------------------------
print("=== 1. Core Aggregate Functions ===")
spark.sql("""
    SELECT
        department,
        COUNT(*)                        AS total_rows,
        COUNT(employee)                 AS non_null_emp,   -- ignores NULLs
        COUNT(DISTINCT employee)        AS distinct_emp,
        SUM(salary)                     AS total_salary,
        AVG(salary)                     AS avg_salary,
        ROUND(AVG(salary), 2)           AS avg_salary_r,
        MIN(salary)                     AS min_salary,
        MAX(salary)                     AS max_salary,
        MAX(salary) - MIN(salary)       AS salary_range,
        STDDEV(salary)                  AS std_dev,        -- sample std dev
        STDDEV_POP(salary)              AS std_dev_pop,    -- population std dev
        VARIANCE(salary)                AS variance,
        COLLECT_LIST(employee)          AS emp_list,       -- ordered list w/ dups
        COLLECT_SET(employee)           AS emp_set,        -- distinct, unordered
        FIRST(employee)                 AS first_emp,
        LAST(employee)                  AS last_emp,
        PERCENTILE(salary, 0.5)         AS median_salary,
        PERCENTILE(salary, ARRAY(0.25, 0.75)) AS quartiles
    FROM sales
    GROUP BY department
    ORDER BY department
""").show(truncate=False)


##### 2. COUNT(*) vs COUNT(col) vs COUNT(DISTINCT col)

In [0]:
# ---------------------------------------------------------------------------
# 2. COUNT(*) vs COUNT(col) vs COUNT(DISTINCT col)
#    KEY EXAM TRAP: COUNT(*) includes NULLs; COUNT(col) ignores NULLs
# ---------------------------------------------------------------------------
print("=== 2. COUNT variants — NULL behavior ===")
spark.sql("""
    SELECT
        COUNT(*)                  AS count_star,       -- 8 (includes NULL customer)
        COUNT(customer)           AS count_col,        -- 7 (excludes NULL)
        COUNT(DISTINCT customer)  AS count_distinct,   -- 4
        COUNT(DISTINCT product)   AS distinct_products -- 5
    FROM orders
""").show()

##### 3. HAVING  — filter on aggregated result (WHERE cannot use aggregates)

In [0]:
# ---------------------------------------------------------------------------
# 3. HAVING  — filter on aggregated result (WHERE cannot use aggregates)
# ---------------------------------------------------------------------------
print("=== 3. GROUP BY + HAVING ===")
spark.sql("""
    SELECT
        department,
        COUNT(*) AS headcount,
        ROUND(AVG(salary), 0) AS avg_sal
    FROM sales
    GROUP BY department
    HAVING headcount >= 2 AND avg_sal > 70000
    ORDER BY avg_sal DESC
""").show()


##### 4. ROLLUP  — hierarchical subtotals

In [0]:
# ---------------------------------------------------------------------------
# 4. ROLLUP  — hierarchical subtotals
#    GROUP BY ROLLUP(a, b)  produces:
#      (a, b) individual groups
#      (a)    subtotal per a  (b = NULL)
#      ()     grand total     (a = NULL, b = NULL)
# ---------------------------------------------------------------------------
print("=== 4. ROLLUP (hierarchical subtotals) ===")
spark.sql("""
    SELECT
        COALESCE(quarter, 'ALL QUARTERS')       AS quarter,
        COALESCE(department, 'ALL DEPTS')       AS department,
        SUM(salary)                             AS total_salary,
        COUNT(*)                                AS headcount
    FROM sales
    GROUP BY ROLLUP(quarter, department)
    ORDER BY quarter, department
""").show(truncate=False)


##### 5. CUBE  — all possible subtotal combinations

In [0]:
# ---------------------------------------------------------------------------
# 5. CUBE  — all possible subtotal combinations
#    GROUP BY CUBE(a, b)  produces:
#      (a, b)   individual groups
#      (a, NULL)  subtotal per a
#      (NULL, b)  subtotal per b
#      (NULL, NULL) grand total
# ---------------------------------------------------------------------------
print("=== 5. CUBE (all combinations) ===")
spark.sql("""
    SELECT
        COALESCE(quarter, 'ALL')     AS quarter,
        COALESCE(department, 'ALL')  AS department,
        SUM(salary)                  AS total_salary
    FROM sales
    GROUP BY CUBE(quarter, department)
    ORDER BY quarter, department
""").show(truncate=False)


##### 6. GROUPING SETS  — explicit control over which combinations

In [0]:
# ---------------------------------------------------------------------------
# 6. GROUPING SETS  — explicit control over which combinations
#    Equivalent to UNION ALL of multiple GROUP BYs (but more efficient)
# ---------------------------------------------------------------------------
print("=== 6. GROUPING SETS ===")
spark.sql("""
    SELECT
        quarter,
        department,
        SUM(salary) AS total_salary
    FROM sales
    GROUP BY GROUPING SETS (
        (quarter, department),  -- detail level
        (quarter),              -- per-quarter total
        (department),           -- per-dept total
        ()                      -- grand total
    )
    ORDER BY quarter, department
""").show(truncate=False)

##### 7. GROUPING() function

In [0]:
# ---------------------------------------------------------------------------
# 7. GROUPING() function
#    Returns 1 if that column is NULL due to ROLLUP/CUBE/GROUPING SETS
#    Returns 0 if it's a real NULL or a real group value
# ---------------------------------------------------------------------------
print("=== 7. GROUPING() — distinguish real NULL from subtotal NULL ===")
spark.sql("""
    SELECT
        quarter,
        department,
        SUM(salary)       AS total_salary,
        GROUPING(quarter) AS q_is_subtotal,   -- 1 = this is a subtotal row
        GROUPING(department) AS d_is_subtotal
    FROM sales
    GROUP BY ROLLUP(quarter, department)
    ORDER BY quarter, department
""").show(truncate=False)


##### 8. APPROX_COUNT_DISTINCT  — HyperLogLog estimation

In [0]:
# ---------------------------------------------------------------------------
# 8. APPROX_COUNT_DISTINCT  — HyperLogLog estimation (faster for big data)
# ---------------------------------------------------------------------------
print("=== 8. APPROX_COUNT_DISTINCT ===")
spark.sql("""
    SELECT
        COUNT(DISTINCT customer)         AS exact_distinct,
        APPROX_COUNT_DISTINCT(customer)  AS approx_distinct,  -- ~5% error
        APPROX_COUNT_DISTINCT(customer, 0.01) AS approx_1pct  -- tighter bound
    FROM orders
""").show()

##### 9. PIVOT  — transform rows into columns

In [0]:
# ---------------------------------------------------------------------------
# 9. PIVOT  — transform rows into columns
# ---------------------------------------------------------------------------
print("=== 9. PIVOT — salary by dept per quarter ===")
spark.sql("""
    SELECT *
    FROM (
        SELECT quarter, department, salary FROM sales
    )
    PIVOT (
        SUM(salary)
        FOR department IN ('Engineering', 'Marketing', 'HR')
    )
    ORDER BY quarter
""").show()

##### 10. Aggregation on multiple GROUP BY columns

In [0]:
# ---------------------------------------------------------------------------
# 10. Aggregation on multiple GROUP BY columns
# ---------------------------------------------------------------------------
print("=== 10. Multi-column GROUP BY with order aggregations ===")
spark.sql("""
    SELECT
        customer,
        COUNT(order_id)                            AS order_count,
        SUM(qty * unit_price)                      AS total_spent,
        ROUND(AVG(qty * unit_price), 2)            AS avg_order_value,
        MIN(order_date)                            AS first_order,
        MAX(order_date)                            AS last_order,
        DATEDIFF(MAX(order_date), MIN(order_date)) AS days_active,
        COLLECT_LIST(product)                      AS products_bought
    FROM orders
    WHERE customer IS NOT NULL
    GROUP BY customer
    ORDER BY total_spent DESC
""").show(truncate=False)


In [0]:
spark.stop()